# InceptionV3 + CBAM — Pneumonia Detection (Third Ensemble Model)
### Companion to pneumonia_notebook_final.ipynb and densenet_notebook.ipynb
**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7

**Why InceptionV3?**
- Parallel multi-scale inception blocks (1×1, 3×3, 5×5 simultaneously)
  — completely different inductive bias from EfficientNet (compound) and DenseNet (dense reuse)
- Native 299×299 input captures finer texture than 260px
- No residual or dense skip connections — different gradient flow, different error patterns
- SEED=0 for maximum diversity vs EN(42) and DN(123)
- Mixup + SGD in Stage 3 — different loss surface from both other models


## Cell 1 — Setup & GPU Check

In [ ]:
# ==================================================
# CELL 1 - SETUP & GPU CHECK (OPTIMIZED FOR RTX 3050)
# Run this first, always
# ==================================================
import os, sys, cv2, shutil, random, time, json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers, mixed_precision
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, Callback)
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam, SGD
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,
                              precision_score, f1_score, confusion_matrix, roc_curve)
from IPython.display import display
import ipywidgets as widgets

# Configure GPU environment for 6GB VRAM
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL']      = '2'

BASE_DIR   = r'C:\pneu\chest_xray'
OUTPUT_DIR = r'C:\pneu\outputs'
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VAL_DIR    = os.path.join(BASE_DIR, 'val')
TEST_DIR   = os.path.join(BASE_DIR, 'test')
os.makedirs(os.path.join(OUTPUT_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'plots'),  exist_ok=True)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Enable Ampere Tensor Core Mixed Precision (1.8x-2.2x speedup + 40% less VRAM)
        mixed_precision.set_global_policy('mixed_float16')
        print(f'GPU : {gpus[0].name} | TF {tf.__version__} | Mixed Precision FP16 Enabled')
    except Exception as e:
        print('GPU setup info:', e)
else:
    print('WARNING: No GPU detected -- running on CPU')

# SEED=0: different from EfficientNet(42) and DenseNet(42) -- maximises ensemble diversity
random.seed(0); np.random.seed(0); tf.random.set_seed(0)

IMG_SIZE = (299, 299)
BATCH    = 8  # Rock-solid stability on 6GB VRAM at 299x299

def _scrub_logs(logs):
    if not logs: return logs
    for k in list(logs.keys()):
        v = logs[k]
        try:
            if hasattr(v, 'numpy'):         logs[k] = float(v.numpy())
            elif isinstance(v, np.ndarray): logs[k] = float(v.flat[0])
            else:                           logs[k] = float(v)
        except Exception: pass
    return logs

_orig_e = tf.keras.callbacks.CallbackList.on_epoch_end
def _patch_e(self, epoch, logs=None):
    return _orig_e(self, epoch, _scrub_logs(logs))
tf.keras.callbacks.CallbackList.on_epoch_end = _patch_e

_orig_b = tf.keras.callbacks.CallbackList.on_train_batch_end
def _patch_b(self, batch, logs=None):
    return _orig_b(self, batch, _scrub_logs(logs))
tf.keras.callbacks.CallbackList.on_train_batch_end = _patch_b

_orig_t = tf.keras.callbacks.CallbackList.on_test_batch_end
def _patch_t(self, batch, logs=None):
    return _orig_t(self, batch, _scrub_logs(logs))
tf.keras.callbacks.CallbackList.on_test_batch_end = _patch_t
print('Callbacks patched')

class PureCheckpoint(Callback):
    def __init__(self, filepath, monitor='val_auc', mode='max', verbose=1):
        super().__init__()
        self.filepath = filepath; self.monitor = monitor
        self.mode = mode; self.verbose = verbose
        self.best = -np.inf if mode == 'max' else np.inf
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val = logs.get(self.monitor)
        if val is None:
            for k, v in logs.items():
                if self.monitor in k:
                    val = v; break
        current = float(val) if val is not None else -np.inf
        improved = current > self.best if self.mode == 'max' else current < self.best
        if improved:
            if self.verbose:
                print(f'  PureCheckpoint: {self.monitor} {self.best:.4f} -> {current:.4f} | saved {os.path.basename(self.filepath)}')
            self.best = current
            self.model.save_weights(self.filepath)

class ProgressBar(Callback):
    def __init__(self, stage, total_epochs, steps):
        super().__init__()
        self.stage = stage; self.total_epochs = total_epochs
        self.steps = steps; self.best_auc = 0.0; self._t0 = None
        try:
            display(widgets.HTML(
                f"<b style='font-size:14px'>Stage {stage} Training</b>"
                f" -- {total_epochs} epochs  |  batch={BATCH}  |  299x299"))
            self.epoch_bar = widgets.IntProgress(value=0, min=0, max=total_epochs,
                description='Epoch 0:', bar_style='info', layout=widgets.Layout(width='55%'))
            self.epoch_lbl = widgets.Label(value='starting...')
            self.best_lbl  = widgets.Label(value='')
            self.batch_bar = widgets.IntProgress(value=0, min=0, max=steps,
                description='Batch:', bar_style='', layout=widgets.Layout(width='55%'))
            self.batch_lbl = widgets.Label(value='')
            display(widgets.HBox([self.epoch_bar, self.epoch_lbl]))
            display(self.best_lbl)
            display(widgets.HBox([self.batch_bar, self.batch_lbl]))
            self.use_widgets = True
        except Exception:
            self.use_widgets = False

    def on_epoch_begin(self, epoch, logs=None):
        self._t0 = time.time()
        if self.use_widgets:
            self.epoch_bar.description = f'Epoch {epoch+1}:'
            self.batch_bar.value = 0

    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        if self.use_widgets:
            self.batch_bar.value = batch + 1
            self.batch_lbl.value = (f"loss={float(logs.get('loss',0)):.4f}  "
                                    f"acc={float(logs.get('accuracy',0)):.4f}")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        elapsed = time.time() - self._t0
        vauc = float(logs.get('val_auc', 0))
        vacc = float(logs.get('val_accuracy', 0))
        if self.use_widgets:
            if vauc > self.best_auc:
                self.best_auc = vauc
                self.best_lbl.value = f'Best val_auc={vauc:.4f} (epoch {epoch+1})'
                self.epoch_bar.bar_style = 'success'
            else:
                self.epoch_bar.bar_style = 'info'
            self.epoch_bar.value = epoch + 1
            self.epoch_lbl.value = (f"val_acc={vacc*100:.2f}%  val_auc={vauc*100:.2f}%  ({elapsed:.0f}s)")
        else:
            print(f"Epoch {epoch+1}/{self.total_epochs} ({elapsed:.0f}s) - loss: {float(logs.get('loss',0)):.4f} - acc: {vacc*100:.2f}% - val_auc: {vauc*100:.2f}%")

print(f'IMG_SIZE={IMG_SIZE}  BATCH={BATCH}  SEED=0')
print('CELL 1 DONE -- proceed to Cell 2')


## Cell 2 — Dataset Check
Just verifies val set is balanced (run pneumonia_notebook_final.ipynb Cell 2 first).

In [ ]:
# ==================================================
# CELL 2 - DATASET CHECK
# Val set must already be balanced by pneumonia_notebook_final Cell 2
# This cell just verifies -- does NOT move files
# ==================================================
print('Dataset split:')
for s, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    n = len(os.listdir(os.path.join(path, 'NORMAL')))
    p = len(os.listdir(os.path.join(path, 'PNEUMONIA')))
    ratio = p / n
    flag  = '  OK (1.67:1 matches test)' if 1.4 <= ratio <= 2.0 else '  WARNING: unexpected ratio'
    print(f'  {s:5}: {n+p:4}  (NORMAL={n}  PNEUMONIA={p}  ratio={ratio:.2f}){flag}')

val_n = len(os.listdir(os.path.join(VAL_DIR, 'NORMAL')))
if val_n < 100:
    raise RuntimeError(
        'Val NORMAL folder looks empty. '
        'Run pneumonia_notebook_final.ipynb Cell 2 first to rebuild the balanced val set.'
    )
print('CELL 2 DONE -- proceed to Cell 3')


## Cell 3 — CLAHE + Generators + Mixup + Class Weights

In [ ]:
# ==================================================
# CELL 3 - CLAHE + GENERATORS + MIXUP + CLASS WEIGHTS
# ==================================================

# Same CLAHE as both other notebooks (grayscale, then merge to 3-channel)
def apply_clahe(image):
    img  = image.astype(np.uint8)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    cl   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(gray)
    return cv2.merge([cl, cl, cl]).astype(np.float32) / 255.0

# Augmentation: different param values from EN and DN for ensemble diversity
train_gen = ImageDataGenerator(
    preprocessing_function=apply_clahe,
    rotation_range=20,
    width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.20, horizontal_flip=True,
    shear_range=0.12, brightness_range=[0.70, 1.30],
    fill_mode='nearest'
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='binary', color_mode='rgb', shuffle=True, seed=0)

val_gen = ImageDataGenerator(preprocessing_function=apply_clahe
).flow_from_directory(VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='binary', color_mode='rgb', shuffle=False)

test_gen = ImageDataGenerator(preprocessing_function=apply_clahe
).flow_from_directory(TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='binary', color_mode='rgb', shuffle=False)

STEPS     = train_gen.samples // BATCH
VAL_STEPS = val_gen.samples   // BATCH

# Mixup: same alpha=0.20 as DenseNet but independent generator (SEED=0)
def mixup_generator(gen, alpha=0.20):
    while True:
        x1, y1 = next(gen)
        x2, y2 = next(gen)
        lam = np.random.beta(alpha, alpha)
        yield lam * x1 + (1 - lam) * x2, lam * y1 + (1 - lam) * y2

train_gen_mix = mixup_generator(train_gen, alpha=0.20)

cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=train_gen.classes)
CLASS_WEIGHT = {0: cw[0], 1: cw[1]}

print(f'Steps/epoch  : {STEPS}  |  Val steps : {VAL_STEPS}')
print(f'Class weights: NORMAL={CLASS_WEIGHT[0]:.2f}  PNEUMONIA={CLASS_WEIGHT[1]:.2f}')
print('CELL 3 DONE -- proceed to Cell 4')


## Cell 4 — Build InceptionV3 + CBAM

In [ ]:
# ==================================================
# CELL 4 - BUILD InceptionV3 + CBAM
# ==================================================
L2 = 1e-4

def channel_attention(x, ratio=8):
    ch = x.shape[-1]
    d1 = layers.Dense(ch//ratio, activation='relu',
                       kernel_regularizer=regularizers.l2(L2))
    d2 = layers.Dense(ch, kernel_regularizer=regularizers.l2(L2))
    avg = layers.GlobalAveragePooling2D()(x)
    mx  = layers.GlobalMaxPooling2D()(x)
    out = layers.Activation('sigmoid')(layers.Add()([d2(d1(avg)), d2(d1(mx))]))
    return layers.Multiply()([x, layers.Reshape((1, 1, ch))(out)])

def spatial_attention(x):
    avg = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    mx  = layers.Lambda(lambda t: tf.reduce_max(t,  axis=-1, keepdims=True))(x)
    out = layers.Conv2D(1, 7, padding='same', activation='sigmoid')(
          layers.Concatenate(axis=-1)([avg, mx]))
    return layers.Multiply()([x, out])

def build_inceptionv3():
    inp  = layers.Input(shape=(*IMG_SIZE, 3))
    base = InceptionV3(include_top=False, weights='imagenet', input_tensor=inp)
    base.trainable = False
    # CBAM: channel-first then spatial (same order as EfficientNet and DenseNet)
    x = channel_attention(base.output)
    x = spatial_attention(x)
    x = layers.GlobalAveragePooling2D()(x)
    for units, drop in [(512, 0.50), (256, 0.40)]:
        x = layers.Dense(units, kernel_regularizer=regularizers.l2(L2))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Dropout(drop)(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32',
                       kernel_regularizer=regularizers.l2(L2))(x)
    return Model(inp, out, name='InceptionV3_CBAM'), base

iv3_model, iv3_base = build_inceptionv3()
print(f'Total params : {iv3_model.count_params():,}')
print(f'Input shape  : {iv3_model.input_shape}')
print('CELL 4 DONE -- proceed to Cell 5')


## Cell 5 — Stage 1: Head Only (backbone frozen, 15 epochs)
Warms up the Dense head before touching backbone weights.

In [ ]:
# ==================================================
# CELL 5 - STAGE 1: Head only | Fast Converge (6 epochs)
# Saves: iv3_stage1.h5
# ==================================================
print('='*55)
print('IV3 STAGE 1 | backbone FROZEN | lr=1e-3 | ~10 min')
print('='*55)

iv3_base.trainable = False
iv3_model.compile(
    optimizer=Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.10),
    metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
)

# Stage 1 uses standard aug (not mixup) + class_weight
train_gen_s1 = ImageDataGenerator(
    preprocessing_function=apply_clahe,
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.20, horizontal_flip=True, shear_range=0.12,
    brightness_range=[0.70, 1.30], fill_mode='nearest'
).flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='binary', color_mode='rgb', shuffle=True, seed=0)

h1 = iv3_model.fit(
    train_gen_s1,
    epochs=6, steps_per_epoch=STEPS,
    validation_data=val_gen, validation_steps=VAL_STEPS,
    class_weight=CLASS_WEIGHT,
    callbacks=[
        PureCheckpoint(os.path.join(OUTPUT_DIR,'models','iv3_stage1.h5'),
                       monitor='val_auc', mode='max', verbose=1),
        EarlyStopping(monitor='val_auc', patience=4,
                      restore_best_weights=True, mode='max', verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=2, min_lr=1e-10, verbose=0),
        ProgressBar(1, 6, STEPS)
    ], verbose=0)

with open(os.path.join(OUTPUT_DIR,'models','iv3_history_s1.json'),'w') as f:
    json.dump({k:[float(v) for v in vals] for k,vals in h1.history.items()}, f, indent=2)
print(f'IV3 Stage 1:  acc={max(h1.history["val_accuracy"])*100:.2f}%   auc={max(h1.history["val_auc"])*100:.2f}%')
print('Saved: iv3_stage1.h5')
print('CELL 5 DONE -- proceed to Cell 6')


## Cell 6 — Stage 2: Top 50% Unfrozen + Mixup (25 epochs)
BN stays frozen. Switches to mixup for smoother decision boundaries.

In [ ]:
# ==================================================
# CELL 6 - STAGE 2: Top 50% unfrozen + Mixup (10 epochs)
# Saves: iv3_stage2.h5
# ==================================================
s1_path = os.path.join(OUTPUT_DIR,'models','iv3_stage1.h5')
if os.path.exists(s1_path):
    iv3_model.load_weights(s1_path)
    print('Loaded weights from iv3_stage1.h5')

print('='*55)
print('IV3 STAGE 2 | top 50% unfrozen, BN frozen | lr=1e-4 | ~15 min')
print('='*55)

iv3_base.trainable = True
cutoff = int(len(iv3_base.layers) * 0.50)
for i, l in enumerate(iv3_base.layers):
    if i < cutoff:
        l.trainable = False
    else:
        l.trainable = False if isinstance(l, tf.keras.layers.BatchNormalization) else True

unfrozen = sum(1 for l in iv3_base.layers if l.trainable)
print(f'Unfrozen: {unfrozen}/{len(iv3_base.layers)} backbone layers  (BN always frozen)')

iv3_model.compile(
    optimizer=Adam(1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
)

h2 = iv3_model.fit(
    train_gen_mix,
    epochs=10, steps_per_epoch=STEPS,
    validation_data=val_gen, validation_steps=VAL_STEPS,
    callbacks=[
        PureCheckpoint(os.path.join(OUTPUT_DIR,'models','iv3_stage2.h5'),
                       monitor='val_auc', mode='max', verbose=1),
        EarlyStopping(monitor='val_auc', patience=4,
                      restore_best_weights=True, mode='max', verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                          patience=2, min_lr=1e-10, verbose=0),
        ProgressBar(2, 10, STEPS)
    ], verbose=0)

with open(os.path.join(OUTPUT_DIR,'models','iv3_history_s2.json'),'w') as f:
    json.dump({k:[float(v) for v in vals] for k,vals in h2.history.items()}, f)
print(f'IV3 Stage 2:  acc={max(h2.history["val_accuracy"])*100:.2f}%   auc={max(h2.history["val_auc"])*100:.2f}%')
print('Saved: iv3_stage2.h5')
print('CELL 6 DONE -- proceed to Cell 7')


## Cell 7 — Stage 3: Full Fine-tune + Curves (25 epochs)
All BN frozen. SGD+Nesterov finds flatter minima than Adam — different error patterns for ensemble.

In [ ]:
# ==================================================
# CELL 7 - STAGE 3: Full fine-tune + curves (10 epochs)
# Saves: iv3_stage3_best.h5 + iv3_final.h5 + iv3_curves.png
# ==================================================
s2_path = os.path.join(OUTPUT_DIR,'models','iv3_stage2.h5')
if os.path.exists(s2_path):
    iv3_model.load_weights(s2_path)
    print('Loaded weights from iv3_stage2.h5')

print('='*55)
print('IV3 STAGE 3 | ALL unfrozen, BN frozen | SGD CosineDecay lr=2e-4 | ~15 min')
print('='*55)

for l in iv3_model.layers:
    l.trainable = False if isinstance(l, tf.keras.layers.BatchNormalization) else True
print(f'BN frozen | Trainable params: {sum([tf.size(w).numpy() for w in iv3_model.trainable_weights]):,}')

S3_EPOCHS = 10
cosine_lr = tf.keras.experimental.CosineDecay(
    initial_learning_rate=2e-4, decay_steps=STEPS * S3_EPOCHS, alpha=1e-7/2e-4)

iv3_model.compile(
    optimizer=SGD(cosine_lr, momentum=0.9, nesterov=True),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
)

h3 = iv3_model.fit(
    train_gen_mix,
    epochs=S3_EPOCHS, steps_per_epoch=STEPS,
    validation_data=val_gen, validation_steps=VAL_STEPS,
    callbacks=[
        PureCheckpoint(os.path.join(OUTPUT_DIR,'models','iv3_stage3_best.h5'),
                       monitor='val_auc', mode='max', verbose=1),
        EarlyStopping(monitor='val_auc', patience=5,
                      restore_best_weights=True, mode='max', verbose=0),
        ProgressBar(3, S3_EPOCHS, STEPS)
    ], verbose=0)

shutil.copy2(os.path.join(OUTPUT_DIR,'models','iv3_stage3_best.h5'),
             os.path.join(OUTPUT_DIR,'models','iv3_final.h5'))
print('Saved: iv3_stage3_best.h5 and iv3_final.h5')

s3 = max(h3.history['val_accuracy'])
with open(os.path.join(OUTPUT_DIR,'models','iv3_history_s3.json'),'w') as f:
    json.dump({k:[float(v) for v in vals] for k,vals in h3.history.items()}, f)
print(f'IV3 Stage 3:  acc={s3*100:.2f}%   auc={max(h3.history["val_auc"])*100:.2f}%')

# Training curves
with open(os.path.join(OUTPUT_DIR,'models','iv3_history_s1.json')) as f: hs1=json.load(f)
with open(os.path.join(OUTPUT_DIR,'models','iv3_history_s2.json')) as f: hs2=json.load(f)
hs3 = {k:[float(v) for v in vals] for k,vals in h3.history.items()}
s1v=max(hs1['val_accuracy']); s2v=max(hs2['val_accuracy'])
e1=len(hs1['accuracy']); e2=e1+len(hs2['accuracy'])
def combine(a,b,c,k): return a[k]+b[k]+c[k]

fig, axes = plt.subplots(2, 2, figsize=(18,12))
fig.suptitle(f'InceptionV3 Curves  S1:{s1v*100:.1f}% -> S2:{s2v*100:.1f}% -> S3:{s3*100:.1f}%',
             fontsize=14, fontweight='bold')
for ax,(tk,vk,title,col) in zip(axes.flat,[
        ('accuracy','val_accuracy','Accuracy','royalblue'),
        ('loss','val_loss','Loss','tomato'),
        ('auc','val_auc','AUC-ROC','seagreen'),
        ('recall','val_recall','Recall','purple')]):
    t=combine(hs1,hs2,hs3,tk); v=combine(hs1,hs2,hs3,vk)
    ep=range(1,len(t)+1)
    ax.plot(ep,t,color=col,linewidth=2,label='Train')
    ax.plot(ep,v,color=col,linewidth=2,linestyle='--',label='Val')
    ax.axvline(x=e1,color='gray',linestyle=':',alpha=0.8,label='S1->S2')
    ax.axvline(x=e2,color='black',linestyle=':',alpha=0.8,label='S2->S3')
    ax.axhline(y=0.95,color='red',linestyle='--',linewidth=1.5,label='95% target')
    ax.set_title(title,fontsize=12,fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'plots','iv3_curves.png'),dpi=150,bbox_inches='tight')
plt.show()
print('Saved: iv3_curves.png')
print('CELL 7 DONE -- InceptionV3 training complete!')
